# 投资组合回测框架使用示例

本 notebook 演示如何使用可扩展的投资组合回测框架

In [1]:
import sys
sys.path.insert(0, '.')

import pandas as pd
import numpy as np

from portfolio_backtest import (
    BacktestEngine,
    RiskParityStrategy,
    MeanVarianceStrategy
)
from portfolio_backtest.visualization import BacktestVisualizer
from portfolio_backtest.utils import load_price_data

## 1. 加载数据

In [2]:
# 加载价格数据
price_df = load_price_data('./market_close.csv')
print(f"数据形状: {price_df.shape}")
print(f"日期范围: {price_df.index.min()} 到 {price_df.index.max()}")
print(f"\n资产列表:")
for col in price_df.columns:
    print(f"  - {col}")

price_df.head()

数据形状: (2447, 5)
日期范围: 2016-01-04 00:00:00 到 2026-01-27 00:00:00

资产列表:
  - 000300.SH
  - HSI.HI
  - SPX.GI
  - AU9999.SGE
  - 中债综合全价总值指数


symbol,000300.SH,HSI.HI,SPX.GI,AU9999.SGE,中债综合全价总值指数
Date,,,,,
2016-01-04,3469.0662,17894.733307,13088.730512,224.77,119.1701
2016-01-05,3478.7797,17815.475776,13142.697399,226.65,119.0748
2016-01-06,3539.8082,17679.059930,12999.184164,228.09,119.1224
2016-01-07,3294.3839,17218.272312,12755.608614,233.24,119.2299
2016-01-08,3361.5632,17312.224681,12615.436108,233.54,119.3380


## 2. 风险平价策略回测

In [3]:
# 创建风险平价策略
rp_strategy = RiskParityStrategy(
    lookback=60,           # 60日回看窗口
    rebalance_freq='ME'    # 月末调仓
)

# 创建回测引擎
engine = BacktestEngine(
    init_cash=1_000_000,
    freq='1D'
)

# 运行回测
rp_result = engine.run(rp_strategy, price_df)

In [4]:
# 查看回测统计
rp_result.stats()

Start                                  2016-01-04 00:00:00
End                                    2026-01-27 00:00:00
Period                                  2447 days 00:00:00
Start Value                                      1000000.0
End Value                                   1561422.904756
Total Return [%]                                  56.14229
Benchmark Return [%]                             152.47663
Max Gross Exposure [%]                               100.0
Total Fees Paid                                        0.0
Max Drawdown [%]                                  7.100992
Max Drawdown Duration                    772 days 00:00:00
Total Trades                                          5208
Total Closed Trades                                   5203
Total Open Trades                                        5
Open Trade PnL                                57708.111711
Win Rate [%]                                      77.70517
Best Trade [%]                                   53.9219

In [5]:
# 可视化结果
rp_viz = BacktestVisualizer(rp_result)
rp_viz.print_metrics()
rp_viz.plot_summary()


Risk Parity 策略表现
总收益率: 56.14%
年化收益率: 6.87%
年化波动率: 4.94%
夏普比率: 1.369
索提诺比率: 2.006
Calmar比率: 0.968
Omega比率: 1.247
最大回撤: -7.10%



In [6]:
# 权重热力图
rp_viz.plot_weights_heatmap(freq='QE')

## 3. 均值方差策略回测

In [7]:
# 创建均值方差策略（最大化夏普比率）
mv_strategy = MeanVarianceStrategy(
    lookback=60,
    rebalance_freq='ME'
)

# 运行回测
mv_result = engine.run(mv_strategy, price_df)

# 可视化
mv_viz = BacktestVisualizer(mv_result)
mv_viz.print_metrics()
mv_viz.plot_summary()


Mean Variance 策略表现
总收益率: 122.61%
年化收益率: 12.68%
年化波动率: 10.65%
夏普比率: 1.174
索提诺比率: 1.717
Calmar比率: 1.157
Omega比率: 1.232
最大回撤: -10.96%



## 4. 策略对比

In [8]:
# 创建多个策略进行对比
strategies = [
    RiskParityStrategy(lookback=60, rebalance_freq='ME'),
    RiskParityStrategy(lookback=120, rebalance_freq='QE'),
    MeanVarianceStrategy(lookback=60, rebalance_freq='ME'),
]

names = ['风险平价(60日/月)', '风险平价(120日/季)', '均值方差(60日/月)']

# 运行所有策略
results = []
for strategy, name in zip(strategies, names):
    result = engine.run(strategy, price_df)
    results.append(result)
    print(f"{name}: 总收益={result.metrics['total_return']*100:.2f}%, 夏普={result.metrics['sharpe_ratio']:.3f}")

风险平价(60日/月): 总收益=43.88%, 夏普=1.080
风险平价(120日/季): 总收益=36.50%, 夏普=0.975
均值方差(60日/月): 总收益=122.61%, 夏普=1.174


In [9]:
# 累计收益对比图
BacktestVisualizer.compare_results(results, names=names)

AttributeError: 'Portfolio' object has no attribute 'cum_returns'

In [ ]:
# 指标对比表
comparison_table = BacktestVisualizer.compare_metrics_table(results, names)
comparison_table

,总收益率 (%),年化收益率 (%),年化波动率 (%),夏普比率,索提诺比率,Calmar比率,最大回撤 (%)
风险平价(60日/月),26.957849,3.517597,6.655376,0.552915,0.748680,0.211500,-16.631631
风险平价(120日/季),24.146976,3.182449,6.380568,0.523236,0.677553,0.287358,-11.074853
均值方差(60日/月),122.608656,12.289315,10.497531,1.156849,1.691542,1.121385,-10.959047


## 5. 创建自定义策略

继承 `BaseStrategy` 即可创建自己的策略

In [ ]:
from portfolio_backtest.strategies.base import BaseStrategy

class EqualWeightStrategy(BaseStrategy):
    """等权重策略 - 自定义策略示例"""
    
    def __init__(self, rebalance_freq='ME'):
        super().__init__(name="Equal Weight", rebalance_freq=rebalance_freq)
        self.rebalance_freq = rebalance_freq
    
    def generate_weights(self, price_df, rebalance_mask=None):
        price_df = self.validate_data(price_df)
        
        if rebalance_mask is None:
            rebalance_dates = self.get_rebalance_dates(price_df, self.rebalance_freq)
            rebalance_mask = pd.Series(
                price_df.index.isin(rebalance_dates),
                index=price_df.index
            )
        
        n_assets = price_df.shape[1]
        equal_weight = 1.0 / n_assets
        
        rebalance_dates = price_df.index[rebalance_mask]
        weights_list = [np.full(n_assets, equal_weight) for _ in rebalance_dates]
        
        return pd.DataFrame(
            weights_list,
            index=rebalance_dates,
            columns=price_df.columns
        )

# 使用自定义策略
ew_strategy = EqualWeightStrategy(rebalance_freq='ME')
ew_result = engine.run(ew_strategy, price_df)

ew_viz = BacktestVisualizer(ew_result)
ew_viz.print_metrics()


Equal Weight 策略表现
总收益率: 132.42%
年化收益率: 12.99%
年化波动率: 10.64%
夏普比率: 1.201
索提诺比率: 1.739
Calmar比率: 0.803
Omega比率: 1.195
最大回撤: -16.19%

